In [2]:
import osmium
import json
import pandas as pd
import geopandas as gpd 

### RailwayStation Centroid

In [ ]:
# 1. Read the railway station (previous data: see the parameter to be considered)

railway_station = gpd.read_file('/Users/sell/work/code_work/daina_project/data/input/dolnoslaskie/railway_station.geojson')

In [ ]:
railway_station.columns

Index(['full_id', 'osm_id', 'osm_type', 'railway', 'addr:city:simc', 'bus',
       'baby_feeding', 'air_conditioning', 'name:zh', 'historic:railway:ref',
       ...
       'opening_date', 'razed:railway', 'construction:platforms',
       'construction:railway', 'fixme', 'layer', 'path', 'x', 'y', 'geometry'],
      dtype='object', length=131)

In [ ]:
railway_station[['osm_id','osm_type','railway','x','y','geometry']]

,osm_id,osm_type,railway,x,y,geometry
0,155479759,node,station,17.038,51.098,POINT (17.03795 51.09792)
1,241763683,node,station,17.084,51.143,POINT (17.08354 51.14255)
2,242360270,node,station,17.297,50.931,POINT (17.29743 50.93075)
3,248216807,node,station,17.034,50.609,POINT (17.03389 50.60865)
4,266565814,node,station,16.486,51.216,POINT (16.48597 51.21569)
...,...,...,...,...,...,...
411,None,None,halt,15.798,50.862,POINT (15.79778 50.86237)
412,None,None,halt,15.794,50.887,POINT (15.79441 50.88682)
413,None,None,halt,16.255,51.215,POINT (16.25463 51.21501)
414,None,None,halt,16.510,50.612,POINT (16.50958 50.61205)


In [19]:
# 2. make the new algorithm for taking the update data 

class RailwayToGeoJSON(osmium.SimpleHandler):
    def __init__(self, bbox=None):
        """
        bbox is optional. 
        If provided, it should be: (min_lat, min_lon, max_lat, max_lon)
        """
        super(RailwayToGeoJSON, self).__init__()
        self.features = []
        self.bbox = bbox

    def node(self, n):
        if 'railway' in n.tags and n.tags['railway'] == 'station':# this is optional > in['station', 'halt']
            lon = n.location.lon
            lat = n.location.lat
            
            # --- Optional Bounding Box Logic ---
            is_inside = True
            if self.bbox:
                min_lat, min_lon, max_lat, max_lon = self.bbox
                if not (min_lat <= lat <= max_lat and min_lon <= lon <= max_lon):
                    is_inside = False
            
            if is_inside:
                # 1. Convert tags to a readable dictionary
                all_tags = {tag.k: tag.v for tag in n.tags}
                
                # 2. Print to console to inspect the data
                station_name = all_tags.get('name', 'Unnamed')
                print(f"--- Station: {station_name} ---")
                print(json.dumps(all_tags, indent=2, ensure_ascii=False))
                print("-" * 30)
                
                feature = {
                    "type": "Feature",
                    "geometry": {
                        "type": "Point",
                        "coordinates": [lon, lat] #  "properties": all_tags
                    },
                    "properties": {
                        "osm_id": n.id,
                        "name": n.tags.get('name', 'null'),
                        "operator": n.tags.get ('operator','null'),
                        "railway": n.tags['railway'],
                        "x": lon,
                        "y": lat
                        
                    }
                }
                self.features.append(feature)



In [20]:
# --- Scenario 1: Using the Bounding Box (Dolnośląskie) ---
dolnoslaskie_bbox = (50.5, 14.5, 51.8, 17.8)
handler_with_bbox = RailwayToGeoJSON(bbox=dolnoslaskie_bbox)
handler_with_bbox.apply_file("/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/poland_260117.osm.pbf")

geojson_data = {
    "type": "FeatureCollection",
    "features": handler_with_bbox.features}
# 3. Save to file
output_path = "/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/railwayStation_centroid_dolnoslaski.geojson"
with open(output_path, 'w') as f:
    json.dump(geojson_data, f, indent=2)

# # --- Scenario 2: No Bounding Box (Whole File/Poland) ---
# handler_all = RailwayToGeoJSON(bbox=None)

--- Station: Wrocław Główny ---
{
  "alt_name": "Wrocław Główny Osobowy;Dworzec PKP Wrocław Główny",
  "name": "Wrocław Główny",
  "network": "PKP Polskie Linie Kolejowe S.A.",
  "operator": "PKP Polskie Linie Kolejowe",
  "operator:short": "PKP PLK",
  "public_transport": "station",
  "railway": "station",
  "railway:ref": "WG",
  "railway:ref:DB": "XPWR",
  "train": "yes",
  "uic_ref": "5100069",
  "wheelchair": "yes",
  "wikidata": "Q428443",
  "wikipedia": "pl:Wrocław Główny"
}
------------------------------
--- Station: Wrocław Sołtysowice ---
{
  "name": "Wrocław Sołtysowice",
  "old_name:de": "Breslau Schottwitz",
  "operator": "PKP Polskie Linie Kolejowe",
  "operator:short": "PKP PLK",
  "public_transport": "station",
  "railway": "station",
  "railway:ref": "WSł",
  "uic_ref": "5104139",
  "wikidata": "Q2529968",
  "wikipedia": "pl:Wrocław Sołtysowice"
}
------------------------------
--- Station: Oława ---
{
  "name": "Oława",
  "operator": "PKP Polskie Linie Kolejowe",
  "o

In [21]:
# read the updated data
railwayStation_centroid = gpd.read_file("/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/railwayStation_centroid_dolnoslaski.geojson")
railwayStation_centroid.head(5)

,osm_id,name,operator,railway,x,y,geometry
0,155479759,Wrocław Główny,PKP Polskie Linie Kolejowe,station,17.037951,51.097916,POINT (17.03795 51.09792)
1,241763683,Wrocław Sołtysowice,PKP Polskie Linie Kolejowe,station,17.083537,51.142548,POINT (17.08354 51.14255)
2,242360270,Oława,PKP Polskie Linie Kolejowe,station,17.297431,50.930748,POINT (17.29743 50.93075)
3,248216807,Ziębice,PKP Polskie Linie Kolejowe,station,17.033887,50.608650,POINT (17.03389 50.60865)
4,249073484,Višňová,Správa železnic,station,15.028598,50.973291,POINT (15.0286 50.97329)


##### Road Network

In [ ]:
# class AllRoadsHandler(osmium.SimpleHandler):
#     def __init__(self, bbox=None):
#         super(AllRoadsHandler, self).__init__()
#         self.features = []
#         self.bbox = bbox # (min_lat, min_lon, max_lat, max_lon)
        
#         self.road_types = {
#             'motorway', 'motorway_link', 'trunk', 'trunk_link', 
#             'primary', 'primary_link', 'secondary', 'secondary_link', 
#             'tertiary', 'tertiary_link', 'unclassified', 'residential', 'service'
#         }

#     def way(self, w):
#         if 'highway' in w.tags and w.tags['highway'] in self.road_types:
#             try:
#                 # 1. Get all coordinates (vertices) for the road
#                 # coords is a list of [longitude, latitude]
#                 coords = [[n.lon, n.lat] for n in w.nodes]
                
#                 # 2. Extract specific start point
#                 start_x = coords[0][0]
#                 start_y = coords[0][1]
                
#                 # 3. Optional Bounding Box Check
#                 is_inside = True
#                 if self.bbox:
#                     min_lat, min_lon, max_lat, max_lon = self.bbox
#                     if not (min_lat <= start_y <= max_lat and min_lon <= start_x <= max_lon):
#                         is_inside = False
                
#                 if is_inside:
#                     # Convert ALL tags to a dictionary
#                     all_tags = {tag.k: tag.v for tag in w.tags}
                    
#                     # Add calculated properties to all_tags first
#                     all_tags["osm_id"] = w.id
#                     all_tags["x_start"] = start_x
#                     all_tags["y_start"] = start_y
#                     # Store the vertices as a list in properties
#                     all_tags["vertices"] = coords 
#                     # Store count of vertices (useful for analysis)
#                     all_tags["vertex_count"] = len(coords)

#                     # Inspect the data (Optional)
#                     # print(f"--- Road: {all_tags.get('name', 'Unnamed')} | Nodes: {len(coords)} ---")

#                     # Selection: define which tags to keep in the GeoJSON properties
#                     keys_to_keep = [
#                         'osm_id', 'name', 'highway', 'surface', 
#                         'x_start', 'y_start', 'vertex_count', 'vertices'
#                     ]
#                     selected_tags = {k: all_tags.get(k, 'null') for k in keys_to_keep}

#                     feature = {
#                         "type": "Feature",
#                         "geometry": {
#                             "type": "LineString",
#                             "coordinates": coords
#                         },
#                         "properties": selected_tags 
#                     }
#                     self.features.append(feature)

#             except osmium.InvalidLocationError:
#                 # Skip if nodes are missing from the PBF extract
#                 pass

# # --- How to Run ---

# # Example: Dolnośląskie BBox
# dolnoslaskie = (51.0, 16.8, 51.2, 17.2)

# # Initialize (Change to bbox=None to get all of Poland)
# handler = AllRoadsHandler(bbox=dolnoslaskie)

# # IMPORTANT: You must use locations=True for road coordinates!
# handler.apply_file("/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/poland_260117.osm.pbf", locations=True)


# # geojson data
# geojson_data = {
#     "type": "FeatureCollection",
#     "features": handler.features}

# # Save
# output_path = "/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/roadLine_network_dolnoslaski_1.geojson"
# with open(output_path, 'w') as f:
#     json.dump(geojson_data, f, indent=2)

# print(f"Extracted {len(handler.features)} road segments.")

In [26]:


class RoadVertexHandler(osmium.SimpleHandler):
    def __init__(self, bbox=None):
        super(RoadVertexHandler, self).__init__()
        self.features = []
        self.bbox = bbox
        self.road_types = {
            'motorway', 'motorway_link', 'trunk', 'trunk_link', 
            'primary', 'primary_link', 'secondary', 'secondary_link', 
            'tertiary', 'tertiary_link', 'unclassified', 'residential', 'service'
        }

    def way(self, w):
        if 'highway' in w.tags and w.tags['highway'] in self.road_types:
            try:
                # 1. Extract all nodes/vertices from the way
                # We use a list of tuples (lon, lat)
                coords = [[n.lon, n.lat] for n in w.nodes]
                
                # 2. Capture all common tags once for duplication
                all_tags = {tag.k: tag.v for tag in w.tags}
                
                # 3. Iterate through every vertex to create individual "Point" rows
                for index, (lon, lat) in enumerate(coords):
                    
                    # 4. Optional BBox Check for the specific vertex
                    is_inside = True
                    if self.bbox:
                        min_lat, min_lon, max_lat, max_lon = self.bbox
                        if not (min_lat <= lat <= max_lat and min_lon <= lon <= max_lon):
                            is_inside = False
                    
                    if is_inside:
                        feature = {
                            "type": "Feature",
                            "geometry": {
                                "type": "Point",
                                "coordinates": [lon, lat]
                            },
                            "properties": {
                                "osm_id": w.id,
                                "name": all_tags.get('name', 'Unnamed'),
                                "highway": all_tags.get('highway', 'null'),
                                "surface": all_tags.get('surface', 'null'),
                                "vertex_index": index,  # Order of point in the road
                                "x":lon,
                                "y":lat,
                                "is_start": index == 0,
                                "is_end": index == len(coords) - 1
                            }
                        }
                        self.features.append(feature)
                        
            except osmium.InvalidLocationError:
                pass

# --- Configuration ---
# Lower Silesia BBox (South, West, North, East)
dolnoslaskie_bbox = (51.0, 16.8, 51.2, 17.2)

handler = RoadVertexHandler(bbox=dolnoslaskie_bbox)
input_pbf = "/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/poland_260117.osm.pbf"

print("Processing... converting road lines into individual vertex points.")
handler.apply_file(input_pbf, locations=True)

# Save to GeoJSON
output_data = {
    "type": "FeatureCollection",
    "features": handler.features
}

output_path = "/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/roadNetwork_centroid_dolnoslaskie.geojson"
with open(output_path, 'w') as f:
    json.dump(output_data, f)

print(f"Done! Created {len(handler.features)} vertex rows.")

Processing... converting road lines into individual vertex points.
Done! Created 290611 vertex rows.


In [24]:
# previous data 
roadLine = gpd.read_file("/Users/sell/work/code_work/daina_project/data/input/dolnoslaskie/road_vertices.geojson")
roadLine.head(5)

,full_id,osm_id,osm_type,path,vertex_index,vertex_part,vertex_part_index,distance,angle,x,y,geometry
0,w4946427,4946427,way,LineString?crs=EPSG:4326&field=full_id:string(...,0,0,0,0.000000,127.697148,15.009,51.180,POINT (15.00934 51.18044)
1,w4946427,4946427,way,LineString?crs=EPSG:4326&field=full_id:string(...,1,0,1,64.712057,127.205659,15.010,51.180,POINT (15.0101 51.18011)
2,w4946427,4946427,way,LineString?crs=EPSG:4326&field=full_id:string(...,2,0,2,232.761476,126.713796,15.012,51.179,POINT (15.01211 51.17927)
3,w4946427,4946427,way,LineString?crs=EPSG:4326&field=full_id:string(...,3,0,3,695.140569,126.712016,15.018,51.177,POINT (15.01761 51.17697)
4,w4946427,4946427,way,LineString?crs=EPSG:4326&field=full_id:string(...,4,0,4,841.847507,126.661077,15.019,51.176,POINT (15.01936 51.17624)


In [25]:
roadLine.columns

Index(['full_id', 'osm_id', 'osm_type', 'path', 'vertex_index', 'vertex_part',
       'vertex_part_index', 'distance', 'angle', 'x', 'y', 'geometry'],
      dtype='object')

In [13]:
roadLine_network = gpd.read_file("/Users/sell/work/code_work/daina_project/data/dataOSM/vector_data/roadLine_network_dolnoslaski.geojson")
roadLine_network.head(5)


,osm_id,name,highway,surface,x_start,y_start,geometry
0,4068291,Luisenstraße,residential,asphalt,14.982784,51.152585,"LINESTRING (14.98278 51.15259, 14.98438 51.153..."
1,4268151,Grüner Graben,residential,asphalt,14.985037,51.155121,"LINESTRING (14.98504 51.15512, 14.98506 51.154..."
2,4268179,Obermarkt,residential,sett,14.986490,51.154906,"LINESTRING (14.98649 51.15491, 14.98663 51.154..."
3,4268180,Klosterplatz,residential,asphalt,14.988064,51.154982,"LINESTRING (14.98806 51.15498, 14.98818 51.154..."
4,4268182,Otto-Buchwitz-Platz,residential,asphalt,14.982784,51.152585,"LINESTRING (14.98278 51.15259, 14.98265 51.152..."


##### Solar Farm

In [18]:
class SolarFullHandler(osmium.SimpleHandler):
    def __init__(self):
        super(SolarFullHandler, self).__init__()
        self.features = []

    def is_solar(self, obj):
        # Check for the specific tags you mentioned
        plant_match = (obj.tags.get('power') == 'plant' and 
                       obj.tags.get('plant:source') == 'solar' and 
                       obj.tags.get('plant:method') == 'photovoltaic')
        
        gen_match = obj.tags.get('generator:source') == 'solar'
        
        return plant_match or gen_match

    def node(self, n):
        if self.is_solar(n):
            self.add_feature(n, "Point", [n.location.lon, n.location.lat], "node")

    def way(self, w):
        # Handle ways that are not closed rings
        if self.is_solar(w) and not w.is_closed():
            try:
                coords = [[n.lon, n.lat] for n in w.nodes]
                self.add_feature(w, "LineString", coords, "way")
            except osmium.InvalidLocationError:
                pass

    def area(self, a):
        # This handles closed Ways (polygons) and Relations (multipolygons)
        if self.is_solar(a):
            try:
                coords = []
                for ring in a.outer_rings():
                    coords.append([[n.lon, n.lat] for n in ring])
                
                # GeoJSON Polygons coordinates
                self.add_feature(a, "Polygon", coords, "area")
            except osmium.InvalidLocationError:
                pass

    def add_feature(self, obj, geom_type, coords, osm_type):
        # Areas use orig_id() because they could be derived from Ways or Relations
        try:
            obj_id = obj.orig_id() if hasattr(obj, 'orig_id') else obj.id
        except:
            obj_id = getattr(obj, 'id', 0)

        feature = {
            "type": "Feature",
            "geometry": {
                "type": geom_type,
                "coordinates": coords
            },
            "properties": {
                "id": obj_id,
                "osm_type_extracted": osm_type,
                "type_tag": obj.tags.get('type', 'null'), # Usually for relations (multipolygon)
                "name": obj.tags.get('name', 'null'),
                "power": obj.tags.get('power', 'null'),
                "plant_method": obj.tags.get('plant:method', 'null'),
                "plant_source": obj.tags.get('plant:source', 'null'),
                "generator_source": obj.tags.get('generator:source', 'null'),
                "output_electricity": obj.tags.get('plant:output:electricity', obj.tags.get('generator:output:electricity', 'null'))
            }
        }
        self.features.append(feature)

In [19]:
# --- Run ---
handler = SolarFullHandler()
input_pbf = "/Users/sell/work/code_work/daina_project/data/dataOSM/poland_260117.osm.pbf"

# locations=True is essential for everything except nodes
handler.apply_file(input_pbf, locations=True)

# Save
output_path = '/Users/sell/work/code_work/daina_project/data/dataOSM/extraction/solar_complete.geojson'
with open(output_path, 'w') as f:
    json.dump({"type": "FeatureCollection", "features": handler.features}, f)

print(f"Total features extracted: {len(handler.features)}")

Total features extracted: 53577
